# Subgroup C-Index Analysis


In [ ]:
library(tidyverse)
library(here)
library(survival)

In [ ]:
# OOF collection helper
source(here('helpers', 'collect_prs_oof_predictions.r'))

# Subgroup analysis helpers
source(here('helpers', 'subgroup_utils.r'))
source(here('helpers', 'subgroup_cindex.r'))

In [ ]:
# Clinical metadata for the pre-defined strata (age, BMI, sex, race/ethnicity).
meta <- read.csv(here("data", "meta_clin.csv"), row.names = 1) %>%
  dplyr::mutate(
    StudyID    = as.character(StudyID),
    sex = as.integer(sex)
  )

In [ ]:
prs_all <- read.csv(here("data", "prs.csv")) %>%
  dplyr::mutate(PGS = stringr::str_remove(PGS, '_hmPOS_GRCh38')) %>%
  dplyr::group_by(PGS) %>%
  dplyr::mutate(SUM = scale(SUM, center = TRUE, scale = TRUE)[, 1]) %>%
  dplyr::ungroup() %>%
  dplyr::distinct()

# Helper: pull a single PGS score and rename to PRS
get_prs <- function(pgs_id) {
  prs_all %>% dplyr::filter(PGS == pgs_id) %>% dplyr::select(StudyID, PRS = SUM)
}

In [ ]:
# define strata for subgroup analyses
strata_configs <- list(
  # age strata
  list(var = 'age_at_sample', cutoff = 60,
       low = 'Age < 60',  high = 'Age > 60',  name = 'Age'),
  # BMI strata
  list(var = 'BMI_interp',    cutoff = 30,
       low = 'BMI < 30',  high = 'BMI > 30',  name = 'BMI'),
  # sex strata
  list(var = 'sex',    cutoff = 0,
       low = 'Female',    high = 'Male',       name = 'Sex'),
  # race/ethnicity strata
  list(var = 'vdw_race_hisp',
       groups = c('Asian' = 'Asian', 'Black' = 'Black',
                  'Hispanic' = 'Hispanic', 'White' = 'White'),
       name = 'Race/Ethnicity'),
  # PRS strata
  list(var = 'PRS', cutoff = 'median',
       low = 'Low PRS', high = 'High PRS', name = 'PRS')
)

In [ ]:
# Marker OOF linear predictors from the primary workflow
# (outputs/main/{prefix}{base|clin|marker}[_prs][_nmr]_oof.rds).
# Prefix encodes the disease + SOC component used (a1c, egfr, fib4, prevent, etc.)
oof_dir <- here("outputs/main")

oof_prefixes <- c(
  t2d        = 't2d_a1c_',
  ckd        = 'ckd_egfr_',
  nafld      = 'nafld_fib4_',
  hf_cm      = 'hf_cm_prevent_ingredients_',
  acutemi    = 'acute_mi_prevent_ingredients_',
  chd        = 'chd_revasc_prevent_ingredients_',
  tia_stroke = 'tia_stroke_prevent_ingredients_',
  total_cvd  = 'total_cvd_prevent_ingredients_',
  afib       = 'afib_chargeaf_ingredients_'
)

model_lists <- lapply(oof_prefixes, function(pfx) collect_prs_oof_predictions(oof_dir, pfx))

In [ ]:
# Maps internal disease ids -> display labels and PGS IDs
subgroup_disease_registry <- list(
  t2d        = list(label = 'T2D',              pgs_id = 'PGS002308'),
  ckd        = list(label = 'CKD',              pgs_id = 'PGS000883'),
  nafld      = list(label = 'MASLD',            pgs_id = 'PGS002283'),
  hf_cm      = list(label = 'Heart Fail',        pgs_id = 'PGS005035'),
  acutemi    = list(label = 'Acute MI',         pgs_id = 'PGS005039'),
  chd        = list(label = 'CHD',              pgs_id = 'PGS005039'),
  tia_stroke = list(label = 'TIA/Stroke',       pgs_id = 'PGS005035'),
  total_cvd  = list(label = 'Total CVD',        pgs_id = 'PGS005035'),
  afib       = list(label = 'AFib',             pgs_id = 'PGS005046')
)

disease_labels <- c(
  t2d        = 'T2D',
  ckd        = 'CKD',
  nafld      = 'MASLD',
  hf_cm      = 'Heart Fail',
  acutemi    = 'Acute MI',
  chd        = 'CHD',
  tia_stroke = 'TIA/Stroke',
  total_cvd  = 'Total CVD',
  afib       = 'AFib'
)

In [ ]:
meta_base <- meta %>%
  dplyr::filter(vdw_race_hisp %in% c('Asian', 'Black', 'Hispanic', 'White'))

disease_configs <- lapply(names(subgroup_disease_registry), function(id) {
  reg <- subgroup_disease_registry[[id]]
  list(
    label          = reg$label,
    metadata_df    = meta_base %>% dplyr::left_join(get_prs(reg$pgs_id), by = 'StudyID'),
    input_list     = model_lists[[id]],
    soc_model      = 'marker_SOC',
    nmr_model      = 'marker_SOC+NMR',
    pgs_id         = reg$pgs_id,
    strata_configs = strata_configs
  )
})
names(disease_configs) <- names(subgroup_disease_registry)

cindex_dfs <- lapply(
  disease_configs,
  subgroup_cindex,
  strata_configs = strata_configs,
  min_n          = 20,
  min_events     = 5,
  verbose        = TRUE
)

# Restrict to diseases with a defined label; keep disease_labels order
cindex_dfs  <- cindex_dfs[intersect(names(disease_labels), names(cindex_dfs))]
disease_ids <- names(cindex_dfs)

# C-index adjacency plot

Stack per-disease subgroup C-index tables, compute the whole-cohort (`All`) C-index from OOF predictions (`marker_SOC` vs `marker_SOC+NMR`), and plot SoC and NMR adjacent within each disease.

In [ ]:
# Paper column order: T2D, CKD, MASLD, Heart Fail, Acute MI, CHD, TIA/Stroke, Total CVD, AFib
plot_disease_ids <- intersect(names(disease_labels), disease_ids)
plot_diseases <- unname(disease_labels[plot_disease_ids])

cindex_stratum_order <- list(
  Age  = c("Age < 60", "Age > 60"),
  BMI  = c("BMI < 30", "BMI > 30"),
  Sex  = c("Female", "Male"),
  `Race/Ethnicity` = c("Asian", "Black", "Hispanic", "White"),
  PRS  = c("Low PRS", "High PRS")
)

# Plot facet strip: wrap after the slash so the row label fits
cindex_stratum_order_plot <- cindex_stratum_order
names(cindex_stratum_order_plot)[
  names(cindex_stratum_order_plot) == "Race/Ethnicity"
] <- "Race/\nEthnicity"

relabel_strat_for_plot <- function(df) {
  df %>%
    dplyr::mutate(
      strat_name = dplyr::recode(
        as.character(.data$strat_name),
        "Race/Ethnicity" = "Race/\nEthnicity"
      )
    )
}

primary_cindex_df <- purrr::map_dfr(
  disease_ids,
  function(disease_id) {
    cindex_dfs[[disease_id]] %>%
      dplyr::mutate(
        disease    = dplyr::recode(disease_id, !!!disease_labels),
        strat_name = as.character(strat_name),
        stratum    = as.character(stratum),
        model      = as.character(model),
        .before    = 1
      )
  }
)

source(here("helpers", "plot_adjacency_cindex.r"))
primary_cindex_df %>% dplyr::count(disease, model) # ensure 12 strata per endpoint-specific model

In [ ]:
# Whole-cohort ("All") C-index from OOF predictions — not typed by hand.
# Same estimator as cv_glmnet_oof / the stored RDS $c_index:
#   survival::concordance(Surv(time, status) ~ I(-lp))
# Computed on the full OOF set for each model (not race-filtered).

oof_cindex_from_df <- function(oof_df) {
  surv_obj <- with(oof_df, survival::Surv(time, status))
  conc     <- survival::concordance(surv_obj ~ I(-lp), data = oof_df)
  cidx     <- unname(conc$concordance)
  se       <- sqrt(conc$var)
  tibble::tibble(
    c_index  = cidx,
    se       = se,
    ci_lower = cidx - 1.96 * se,
    ci_upper = cidx + 1.96 * se,
    n        = nrow(oof_df),
    n_events = as.integer(sum(oof_df$status, na.rm = TRUE))
  )
}

global_models <- c("marker_SOC", "marker_SOC+NMR")

global_cindex_df <- purrr::map_dfr(disease_ids, function(disease_id) {
  purrr::map_dfr(global_models, function(model_name) {
    oof_df <- model_lists[[disease_id]][[model_name]]
    if (is.null(oof_df)) {
      stop(sprintf("Missing model '%s' for disease '%s'", model_name, disease_id))
    }
    oof_cindex_from_df(oof_df) %>%
      dplyr::mutate(
        disease         = dplyr::recode(disease_id, !!!disease_labels),
        stratum         = NA_character_,
        strat_name      = "All",
        model           = model_name,
        stratum_n_label = paste0("All (n=", n, ")"),
        .before         = 1
      )
  })
})

global_cindex_df %>%
  dplyr::select(disease, model, c_index, ci_lower, ci_upper, n, n_events)

In [ ]:
plot_cindex_df <- dplyr::bind_rows(primary_cindex_df, global_cindex_df)

cindex_plot <- plot_adjacency_cindex(
  cindex_df      = relabel_strat_for_plot(plot_cindex_df),
  diseases       = plot_diseases,
  drop_strata    = NULL,
  stratum_order  = cindex_stratum_order_plot,
  no_title       = TRUE,
  subtitle       = NULL
)

options(repr.plot.width = 10.5, repr.plot.height = 4)
cindex_plot

ggplot2::ggsave(
  filename = here("outputs/figures", "FIG_3.png"),
  plot     = cindex_plot,
  width    = 10.5, height = 4, units = "in", dpi = 600
)

# Sample-size summary
Per-stratum `n` and event counts (including the whole-cohort `All` row).

In [ ]:
# summarize tte events across strata
subgroup_strata_n <- plot_cindex_df %>%
  # distinct strata
  dplyr::distinct(disease, strat_name, stratum, n, n_events) %>%
  # rename na to all
  dplyr::mutate(
    stratum     = dplyr::if_else(is.na(stratum), "All", stratum),
    count_label = paste0(scales::comma(n), " / ", scales::comma(n_events)),
    strat_name  = factor(
      strat_name,
      levels = c("All", "Age", "BMI", "Sex", "Race/Ethnicity", "PRS")
    )
  ) %>%
  # select columns
  dplyr::select(disease, strat_name, stratum, count_label) %>%
  tidyr::pivot_wider(names_from = disease, values_from = count_label) %>%
  dplyr::arrange(strat_name, stratum)

subgroup_strata_n
write.csv(subgroup_strata_n, here("outputs/tables", "SUBGROUP_STRATA.csv"))

# Disease-specific insets
Within-stratum SoC vs NMR C-index dumbbells for T2D, CKD, and MASLD. Dashed reference lines are the whole-cohort (`All`) C-index from `global_cindex_df`.

In [ ]:
source(here::here("helpers", "plot_within_stratum_cindex_dumbbell.r"))

global_c_for <- function(disease_id, model_name) {
  label <- unname(disease_labels[disease_id])
  vals <- global_cindex_df %>%
    dplyr::filter(.data$disease == label, .data$model == model_name) %>%
    dplyr::pull(.data$c_index)
  if (length(vals) != 1L) {
    stop(sprintf(
      "Expected 1 global C-index for %s / %s, got %d",
      label, model_name, length(vals)
    ))
  }
  vals[[1]]
}

plot_disease_inset <- function(disease_id, title, xlim, outfile, show_legend = FALSE) {
  p <- plot_within_stratum_cindex_dumbbell(
    cidx            = relabel_strat_for_plot(cindex_dfs[[disease_id]]),
    soc_model       = "marker_SOC",
    nmr_model       = "marker_SOC+NMR",
    soc_global      = global_c_for(disease_id, "marker_SOC"),
    nmr_global      = global_c_for(disease_id, "marker_SOC+NMR"),
    title           = title,
    point_size      = 1.5,
    x_by            = 0.10,
    xlim            = xlim,
    model_labels    = c("SOC", "SOC+NMR"),
    legend_position = if (isTRUE(show_legend)) "bottom" else "none"
  )

  ggplot2::ggsave(
    filename = here("outputs/figures", outfile),
    plot     = p,
    width    = 2.25,
    height   = if (isTRUE(show_legend)) 3.35 else 3,
    units    = "in",
    dpi      = 600
  )
  p
}

options(repr.plot.height = 3, repr.plot.width = 2.25) 

In [ ]:
p_t2d_within_stratum <- plot_disease_inset(
  disease_id  = "t2d",
  title       = "T2D",
  xlim        = c(0.60, 0.90),
  outfile     = "FIG3_t2d_inset.png",
  show_legend = FALSE
)
p_t2d_within_stratum

In [ ]:
p_ckd_within_stratum <- plot_disease_inset(
  disease_id = "ckd",
  title      = "CKD",
  xlim       = c(0.60, 0.90),
  outfile    = "FIG3_ckd_inset.png"
)
p_ckd_within_stratum

In [ ]:
p_masld_within_stratum <- plot_disease_inset(
  disease_id = "nafld",
  title      = "MASLD",
  xlim       = c(0.55, 0.75),
  outfile    = "FIG3_masld_inset.png"
)
p_masld_within_stratum

## Incidence (risk enrichment subgroup analysis)
Observed 5-year incidence in the top 5% predicted-risk group, per stratum × disease. Whole-cohort (`All`) incidence is computed from OOF predictions (`marker_SOC` vs `marker_SOC+NMR`), not typed by hand.

In [ ]:
# Top-X% predicted-risk observed incidence per stratum x disease
source(here("helpers", "subgroup_top_risk_incidence_calculator.r"))

top_props <- c(0.01, 0.05, 0.10)

top_dfs <- lapply(
  disease_configs,
  subgroup_top_risk_incidence_calculator,
  strata_configs = strata_configs,
  top_props      = top_props,
  t_star         = 5,
  recalibrate    = FALSE,
  min_n          = 100,
  min_events     = 10,
  verbose        = TRUE
)

top_disease_ids <- intersect(disease_ids, names(top_dfs)) 


In [ ]:
primary_incidence_df <- purrr::map_dfr(
  top_disease_ids,
  function(disease_id) {
    top_dfs[[disease_id]] %>%
      dplyr::mutate(
        disease    = dplyr::recode(disease_id, !!!disease_labels),
        strat_name = as.character(strat_name),
        stratum    = as.character(stratum),
        model      = as.character(model),
        top_label  = as.character(top_label),
        .before    = 1
      )
  }
)

primary_incidence_df %>% dplyr::count(disease, top_label)

In [ ]:
primary_incidence_df %>%
  dplyr::filter(top_label == "Top 5%") %>%
  dplyr::group_by(disease) %>%
  dplyr::summarize(
    min    = min(obs_incidence, na.rm = TRUE),
    max    = max(obs_incidence, na.rm = TRUE),
    q1     = quantile(obs_incidence, 0.25, na.rm = TRUE),
    q3     = quantile(obs_incidence, 0.75, na.rm = TRUE),
    mean   = mean(obs_incidence, na.rm = TRUE),
    median = median(obs_incidence, na.rm = TRUE),
    .groups = "drop"
  )

In [ ]:
# Whole-cohort ("All") top-risk incidence from OOF predictions — not typed by hand.
global_models <- c("marker_SOC", "marker_SOC+NMR")

global_incidence_df <- purrr::map_dfr(top_disease_ids, function(disease_id) {
  res <- top_risk_incidence_calculator(
    input               = model_lists[[disease_id]],
    disease_name        = unname(disease_labels[disease_id]),
    model_keys          = global_models,
    model_labels        = global_models,
    reference_model_key = "marker_SOC",
    top_props           = top_props,
    t_star              = 5,
    recalibrate         = FALSE,
    return_wide         = FALSE,
    verbose             = FALSE
  )

  as.data.frame(res) %>%
    dplyr::mutate(
      disease    = dplyr::recode(disease_id, !!!disease_labels),
      stratum    = NA_character_,
      strat_name = "All",
      model      = as.character(model_key),
      top_label  = as.character(top_label),
      .before    = 1
    )
})

global_incidence_df %>%
  dplyr::filter(top_label == "Top 5%") %>%
  dplyr::select(disease, model, obs_incidence, obs_lower, obs_upper, n_total, n_top)

In [ ]:
source(here("helpers", "plot_adjacency_incidence.r"))

plot_incidence_df <- dplyr::bind_rows(primary_incidence_df, global_incidence_df)

incidence_plot_diseases <- intersect(
  plot_diseases,
  unique(as.character(plot_incidence_df$disease))
)

incidence_plot_top5 <- plot_adjacency_incidence(
  incidence_df   = relabel_strat_for_plot(plot_incidence_df),
  top_label      = "Top 5%",
  diseases       = incidence_plot_diseases,
  disease_labels = NULL,
  drop_strata    = NULL,
  stratum_order  = cindex_stratum_order_plot,
  model_labels   = c("marker_SOC" = "SOC", "marker_SOC+NMR" = "NMR"),
  show_values    = TRUE,
  band_breaks    = c(-Inf, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, Inf),
  band_labels    = c("< 10%", "10-20%", "20-30%", "30-40%", "40-50%", "50-60%", "> 60%"),
  band_colours   = c("#F7F4EE", "#EDEAE4", "#D9E8C9", "#B7DBA8", "#8FCBB0", "#39969D", "#145A32")
)

options(repr.plot.width = 10, repr.plot.height = 4)
incidence_plot_top5

ggplot2::ggsave(
  filename = here("outputs/figures", "[SUBGROUPS_ENRICH_CALIB]_incidence.png"),
  plot     = incidence_plot_top5,
  width    = 10, height = 4, units = "in", dpi = 600
)


# Calibration (subgroup analysis)
Produces:
- Per-stratum ICI for SoC vs SoC+NMR
- Whole-cohort (`All`) ICI from the same OOF risks
- Calibration adequacy adjacency plot (raw ICI in percentage points; lower is better)

In [ ]:
# Fold-specific OOF absolute risks from the primary workflow
# (do not recompute from LPs). Each RDS is the object returned by
# compute_fold_specific_oof_absolute_risk(), with pred_risk_5yr on $oof_predictions.
source(here("helpers", "calibration_from_oof_risk.r"))
source(here("helpers", "subgroup_calibration_from_oof_risk.r"))

abs_risk_dir <- here("outputs/main", "absolute_risks")

load_fold_baseline_pair <- function(prefix) {
  soc_path <- file.path(abs_risk_dir, paste0(prefix, "marker_fold_baseline_risk.rds"))
  nmr_path <- file.path(abs_risk_dir, paste0(prefix, "marker_nmr_fold_baseline_risk.rds"))
  missing <- c(soc_path, nmr_path)[!file.exists(c(soc_path, nmr_path))]
  if (length(missing) > 0) {
    warning(
      "Skipping ", prefix, ". Missing fold-baseline file(s): ",
      paste(basename(missing), collapse = ", ")
    )
    return(NULL)
  }
  list(
    `marker_SOC`     = readRDS(soc_path),
    `marker_SOC+NMR` = readRDS(nmr_path)
  )
}

# Same disease prefixes as the LP OOF collection above
abs_risk_lists <- lapply(oof_prefixes, load_fold_baseline_pair)
names(abs_risk_lists)

vapply(abs_risk_lists, function(x) !is.null(x), logical(1))

In [ ]:
# Raw ICI per stratum x disease from pred_risk_5yr.
# Reuse the c-index/incidence configs, but point input_list at fold-baseline risks.
cal_disease_configs <- lapply(names(disease_configs), function(id) {
  cfg <- disease_configs[[id]]
  cfg$input_list <- abs_risk_lists[[id]]
  cfg
})
names(cal_disease_configs) <- names(disease_configs)

cal_disease_configs <- cal_disease_configs[
  !vapply(cal_disease_configs, function(cfg) is.null(cfg$input_list), logical(1))
]

cal_dfs <- lapply(
  cal_disease_configs,
  subgroup_calibration_from_oof_risk,
  strata_configs = strata_configs,
  t_star         = 5,
  risk_col       = "pred_risk_5yr",
  n_groups       = 10,
  min_n          = 100,
  min_events     = 20,
  verbose        = TRUE
)

cal_disease_ids <- intersect(disease_ids, names(cal_dfs))

In [ ]:
# Stack per-disease calibration tables into one long df (mirrors primary_cindex_df / primary_incidence_df)
primary_calibration_df <- purrr::map_dfr(
  cal_disease_ids,
  function(disease_id) {
    cal_dfs[[disease_id]] %>%
      dplyr::mutate(
        disease    = dplyr::recode(disease_id, !!!disease_labels),
        strat_name = as.character(strat_name),
        stratum    = as.character(stratum),
        model      = as.character(model),
        .before    = 1
      )
  }
)

primary_calibration_df %>% dplyr::count(disease, model)

In [ ]:
primary_calibration_df %>%
  dplyr::group_by(disease) %>%
  dplyr::summarize(
    min    = min(ici, na.rm = TRUE),
    max    = max(ici, na.rm = TRUE),
    q1     = quantile(ici, 0.25, na.rm = TRUE),
    q3     = quantile(ici, 0.75, na.rm = TRUE),
    mean   = mean(ici, na.rm = TRUE),
    median = median(ici, na.rm = TRUE),
    .groups = "drop"
  )

primary_calibration_df %>% dplyr::select(ici, calibration_score) %>% summary()

In [ ]:
# Whole-cohort ("All") ICI from precomputed 5-year OOF risks — not typed by hand.
# Same estimator as compute_calibration_df_from_oof_risk().
# Computed on the full OOF set for each model (not race-filtered).
global_models <- c("marker_SOC", "marker_SOC+NMR")

global_calibration_df <- purrr::map_dfr(cal_disease_ids, function(disease_id) {
  purrr::map_dfr(global_models, function(model_name) {
    res <- abs_risk_lists[[disease_id]][[model_name]]
    if (is.null(res)) {
      stop(sprintf("Missing fold-baseline object '%s' for disease '%s'",
                   model_name, disease_id))
    }
    cal <- compute_calibration_df_from_oof_risk(
      res,
      t_star   = 5,
      risk_col = "pred_risk_5yr",
      n_groups = 10,
      verbose  = FALSE
    )
    tibble::tibble(
      disease           = dplyr::recode(disease_id, !!!disease_labels),
      stratum           = NA_character_,
      strat_name        = "All",
      model             = model_name,
      t_star            = 5,
      ici               = attr(cal, "ici"),
      calibration_score = attr(cal, "calibration_score"),
      n                 = as.integer(attr(cal, "n")),
      n_events          = as.integer(attr(cal, "n_events")),
      stratum_n_label   = paste0("All (n=", attr(cal, "n"), ")")
    )
  })
})

global_calibration_df %>%
  dplyr::select(disease, model, ici, calibration_score, n, n_events)

In [ ]:
# Plot subgroup + global "All" band
source(here("helpers", "plot_adjacency_calibration.r"))

plot_calibration_df <- dplyr::bind_rows(primary_calibration_df, global_calibration_df)

calibration_plot_diseases <- intersect(
  plot_diseases,
  unique(as.character(plot_calibration_df$disease))
)

calibration_plot <- plot_calibration_adequacy_heatmap(
  calibration_df = relabel_strat_for_plot(plot_calibration_df),
  diseases       = calibration_plot_diseases,
  drop_strata    = NULL,
  stratum_order  = cindex_stratum_order_plot,
  model_labels   = c("marker_SOC" = "SOC", "marker_SOC+NMR" = "NMR"),
  show_values    = TRUE,
  metric         = "absolute"  # raw ICI in pp; lower is better
)

options(repr.plot.width = 10, repr.plot.height = 4)
calibration_plot

ggplot2::ggsave(
  filename = here("outputs/figures", "[SUBGROUPS_ENRICH_CALIB]_calibration.png"),
  plot     = calibration_plot,
  width    = 10, height = 4, units = "in", dpi = 600
)